# Prompt Baseline

In [ ]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib/libnccl.so.2")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
nccl_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path + ":" + nccl_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


## 1. Load Data & Base Model

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")

In [ ]:
# === SELECT MODEL ===
MODEL_ID = "ibm-granite/granite-20b-functioncalling"

In [ ]:
# === LOAD MODEL ===
COMPUTE_DTYPE = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)
 
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
model.eval()
print("Model loaded successfully!")

## 2. Evaluate Baseline Full Information Prompt

In [ ]:
# Evaluate on Full Info Dataset
acc_full, results_full = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_full,
    training_strategy="baseline",
    prompt_format="FullInfo",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


## 3. Evaluate Baseline Structural-Only Prompt

In [ ]:
# Evaluate on Structural Only Dataset
acc_struct, results_struct = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_struct,
    training_strategy="baseline",
    prompt_format="structOnly",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


In [ ]:
# === VERIFY EXPERIMENT LOGGING ===
import os
import pandas as pd
from IPython.display import display

csv_path = f"{ARTIFACTS_DIR}/experiment_summary.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display_rows = df[df["Model"] == MODEL_ID]
    print(f"Logged results in {csv_path} for {MODEL_ID}:")
    display(display_rows)
else:
    print(f"⚠️ CSV file not found: {csv_path}")